# Real-Time Fraud Detection — Feature Store + Online Inference

This notebook stands up the **article-aligned** ML backbone on top of your existing demo, keeping **Snowflake Postgres** as the app's system of record.

```
Next.js app  --write-->  Snowflake Postgres  --managed mirror-->  raw TRANSACTIONS (Snowflake)
     ^                    (system of record)                              |
     | POST /predict                                                      v  Dynamic Table
     | (TRANSACTION_ID only)                                       TRANSACTION_FEATURES
     |                                                                    |
     +---- SPCS service <-- XGBoost (Model Registry) <-- Online Feature Store (Postgres, 10s lag)
```

**What this replaces from the current build:** the pg_lake -> Iceberg -> catalog auto-refresh chain (now a Snowflake-managed mirror), and the Cortex Anomaly Detection model + impossible-travel rule in TypeScript (now an XGBoost model whose features are derived once in SQL).

**Run this in a Snowflake Notebook with Container Runtime (CPU).** Warehouse: X-Small or larger.

## Prerequisites

- Snowflake account with **ACCOUNTADMIN** (or a role with feature-store, compute pool, and model registry privileges).
- The managed mirror is **already set up**: `FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS` is kept in sync from your Snowflake Postgres system of record. This notebook only reads from it.
- `snowflake-ml-python >= 1.44.0`.

In [17]:
%pip install snowflake-ml-python httpx -U -q

Note: you may need to restart the kernel to use updated packages.


## Configuration

Edit these to match your environment (warehouse name, database/schema for ML objects).

In [8]:
# === Existing mirrored source (Postgres -> Snowflake already set up) ===
# FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS is the Snowflake table maintained by an
# existing managed mirror from your Snowflake Postgres system of record.
RAW_TABLE = "FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS"

# === Feature Store / model / service ===
FS_DATABASE          = "FRAUD_ML"
FS_SCHEMA            = "FEATURE_STORE"
FEATURE_TABLE        = f"{FS_DATABASE}.{FS_SCHEMA}.CARD_FEATURES"
TRAIN_TABLE          = f"{FS_DATABASE}.{FS_SCHEMA}.FRAUD_TRAINING"
FEATURE_VIEW_NAME    = "CARD_FRAUD_FEATURES"
FEATURE_VIEW_VERSION = "V1"
MODEL_NAME           = "FRAUD_XGBOOST"
MODEL_VERSION        = "V1"
SERVICE_NAME         = "FRAUD_DETECTION_SVC"
COMPUTE_POOL_NAME    = "ML_ONLINE_CPU_POOL"
WAREHOUSE            = "COMPUTE_WH"

PRODUCER_ROLE = "ACCOUNTADMIN"
CONSUMER_ROLE = "ACCOUNTADMIN"

from snowflake.snowpark import Session

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local: build session from ~/.snowflake/connections.toml
    session = Session.builder.configs({"connection_name": "ereitberger-demo-account-aws-eu-frankfurt"}).create()

session.sql(f"USE ROLE {PRODUCER_ROLE}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
print("Session ready as", session.get_current_role())

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Session ready as "ACCOUNTADMIN"


## Part I — Source: existing managed mirror

`FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS` is already maintained by a managed mirror (`SNOWFLAKE.POSTGRES.CREATE_MIRROR`) from your Snowflake Postgres system of record — so there is nothing to create here. The app keeps writing to Postgres; Snowflake keeps this table continuously in sync. We just read from it.

The sanity check below confirms the table is populated before we build features on top of it.

In [10]:
# Sanity check: confirm the mirrored source table is populated.
session.sql(f"SELECT COUNT(*) AS ROW_COUNT FROM {RAW_TABLE}").show()
session.table(RAW_TABLE).select(
    "ID", "CARD_NUMBER", "AMOUNT", "MERCHANT_CATEGORY",
    "LATITUDE", "LONGITUDE", "CREATED_AT", "FRAUD_FLAG"
).show(5)

---------------
|"ROW_COUNT"  |
---------------
|222          |
---------------

-----------------------------------------------------------------------------------------------------------------------------------------
|"ID"  |"CARD_NUMBER"     |"AMOUNT"  |"MERCHANT_CATEGORY"  |"LATITUDE"  |"LONGITUDE"  |"CREATED_AT"                      |"FRAUD_FLAG"  |
-----------------------------------------------------------------------------------------------------------------------------------------
|8     |379628498037781   |618.23    |Travel               |-23.5505    |-46.6333     |2026-06-18 21:07:29.129000-08:00  |False         |
|9     |342439278037808   |61.84     |Fuel                 |41.8781     |-87.6298     |2026-06-20 11:56:12.206000-08:00  |False         |
|10    |5962009116404959  |81.83     |Fuel                 |48.8566     |2.3522       |2026-06-20 22:51:52.614000-08:00  |False         |
|11    |4520590383594336  |1849.24   |Jewelry              |41.8781     |-87.6298     |2026

## Part II — Card feature table (Dynamic Table)

`CARD_FEATURES` holds **one row per card**: the card's current state, derived from its history in the mirrored table. This is what the online store serves at scoring time, keyed by `CARD_NUMBER`.

Columns:
- `CARD_NUMBER` — key
- `HOME_LAT`, `HOME_LON` — card's first-seen location
- `LAST_LAT`, `LAST_LON` — card's most recent location
- `AVG_AMOUNT` — average transaction amount
- `TXN_COUNT` — number of transactions

Because it aggregates the whole card history, the transaction being scored is not yet in it — so the app can look up card state and score a new charge before committing it.

In [11]:
session.sql(f"CREATE DATABASE IF NOT EXISTS {FS_DATABASE}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {FS_DATABASE}.{FS_SCHEMA}").collect()

session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE {FEATURE_TABLE}
  TARGET_LAG = '1 minute'
  WAREHOUSE  = {WAREHOUSE}
AS
SELECT
  CARD_NUMBER,
  MIN_BY(LATITUDE,  CREATED_AT) AS HOME_LAT,
  MIN_BY(LONGITUDE, CREATED_AT) AS HOME_LON,
  MAX_BY(LATITUDE,  CREATED_AT) AS LAST_LAT,
  MAX_BY(LONGITUDE, CREATED_AT) AS LAST_LON,
  AVG(AMOUNT)                   AS AVG_AMOUNT,
  COUNT(*)                      AS TXN_COUNT
FROM {RAW_TABLE}
GROUP BY CARD_NUMBER
""").collect()
print("Dynamic Table", FEATURE_TABLE, "created.")
session.table(FEATURE_TABLE).show(5)

Dynamic Table FRAUD_ML.FEATURE_STORE.CARD_FEATURES created.
----------------------------------------------------------------------------------------------------
|"CARD_NUMBER"    |"HOME_LAT"  |"HOME_LON"  |"LAST_LAT"  |"LAST_LON"  |"AVG_AMOUNT"  |"TXN_COUNT"  |
----------------------------------------------------------------------------------------------------
|342439278037808  |41.8781     |-87.6298    |41.8781     |-87.6298    |260.02714286  |7            |
|343135561393975  |52.52       |13.405      |40.7128     |-74.006     |229.99000000  |4            |
|343318350794267  |40.7128     |-74.006     |40.7128     |-74.006     |557.32000000  |4            |
|344825084204084  |25.2048     |55.2708     |25.2048     |55.2708     |631.99666667  |6            |
|345003818505268  |51.5074     |-0.1278     |51.5074     |-0.1278     |169.45142857  |7            |
----------------------------------------------------------------------------------------------------



## Part III — Feature Store with online serving

Register the `TRANSACTION` entity and a `FeatureView` whose source is the Dynamic Table. The Postgres-backed **online store** (10s target lag) is what serves features at low latency to the inference service.

In [12]:
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode, OnlineConfig, OnlineStoreType
)

fs = FeatureStore(
    session=session,
    database=FS_DATABASE,
    name=FS_SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

card_entity = Entity(name="CARD", join_keys=["CARD_NUMBER"])
fs.register_entity(card_entity)
print("Entity registered:", card_entity.name)

Entity registered: CARD


In [13]:
# Provision the Postgres-backed online serving infrastructure (first run takes a few minutes).
import time

try:
    result = fs.create_online_service(producer_role=PRODUCER_ROLE, consumer_role=CONSUMER_ROLE)
    print(f"Create online service: {result}")
except Exception as e:
    print(f"Online service already exists or error: {e}")

for i in range(60):
    status = fs.get_online_service_status()
    current = status.status if hasattr(status, "status") else str(status)
    print(f"[{i+1}] Online service status: {current}")
    if current == "RUNNING":
        print("Online service is ready!")
        break
    time.sleep(10)
else:
    raise TimeoutError("Service did not reach RUNNING within 10 min.")

Create online service: OnlineServiceResult(status='SUCCESS', message='Online Service created. Poll get_online_service_status() until RUNNING.')
[1] Online service status: STARTING
[2] Online service status: STARTING
[3] Online service status: STARTING
[4] Online service status: STARTING
[5] Online service status: STARTING
[6] Online service status: STARTING
[7] Online service status: STARTING
[8] Online service status: STARTING
[9] Online service status: STARTING
[10] Online service status: STARTING
[11] Online service status: RUNNING
Online service is ready!


In [14]:
source_df = session.table(FEATURE_TABLE).select(
    "CARD_NUMBER", "HOME_LAT", "HOME_LON", "LAST_LAT", "LAST_LON",
    "AVG_AMOUNT", "TXN_COUNT"
)

card_fv = FeatureView(
    name=FEATURE_VIEW_NAME,
    entities=[card_entity],
    feature_df=source_df,
    desc="Card-level fraud detection features",
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
)

card_fv = fs.register_feature_view(feature_view=card_fv, version=FEATURE_VIEW_VERSION, overwrite=True)
print(f"Feature View registered: {card_fv.name} version {card_fv.version} (online store: POSTGRES)")

POSTGRES online store type is in Public Preview since 1.43.0.


Feature View registered: CARD_FRAUD_FEATURES version V1 (online store: POSTGRES)


In [27]:
# Verify low-latency online retrieval. Online client auths via a PAT; the session token works for dev.
import os
from snowflake.ml.feature_store.feature_view import StoreType

os.environ["SNOWFLAKE_PAT"] = session.connection.rest.token

card_fv = fs.get_feature_view(FEATURE_VIEW_NAME, FEATURE_VIEW_VERSION)
print(f"Store type: {card_fv.online_config.store_type}, target lag: {card_fv.online_config.target_lag}")

# Pick a few real card numbers from the feature table
sample_keys = [[r["CARD_NUMBER"]] for r in session.table(FEATURE_TABLE).select("CARD_NUMBER").limit(3).collect()]
online_result = fs.read_feature_view(card_fv, keys=sample_keys, store_type=StoreType.ONLINE)# Install missing dependency required for online feature store access
print("Online feature retrieval OK:")
online_result

Store type: OnlineStoreType.POSTGRES, target lag: 10 seconds


Online Service HTTP error: status=500 request_id=52e8c9b7-4f49-4b37-a51e-49fc11f69d90 body='{"detail":"","responseType":"FAULT","requestId":"52e8c9b7-4f49-4b37-a51e-49fc11f69d90"}'


RuntimeError: (1500) Online Service error (HTTP 500): {"detail":"","responseType":"FAULT","requestId":"52e8c9b7-4f49-4b37-a51e-49fc11f69d90"} [request_id: 52e8c9b7-4f49-4b37-a51e-49fc11f69d90]

In [28]:
fs.get_online_service_status()

OnlineServiceStatus(status='RUNNING', message='Feature Store Online Service is running', endpoints=(OnlineServiceEndpoint(name='ingest', url='https://en5jie-sfseeurope-ereitberger-aws2.snowflakecomputing.app', privatelink_url=None, internal_url='http://fs-runtime-71465480.bjtz.svc.spcs.internal:8080'), OnlineServiceEndpoint(name='query', url='https://in5jie-sfseeurope-ereitberger-aws2.snowflakecomputing.app', privatelink_url=None, internal_url='http://fs-runtime-71465480.bjtz.svc.spcs.internal:8081')), created_at='1787651513033', updated_at='1787653521474')

## Part IV — Training table + XGBoost

Build the training table by joining each historical transaction (its charge attributes) to the card's state in `CARD_FEATURES`, with the existing `FRAUD_FLAG` as the label. Train a plain XGBoost classifier on the same 10 columns the service will see at scoring time (card state from lookup + charge attributes from the request).

Note: this join uses current card state, so there is mild label leakage — acceptable for a prototype whose labels already come from rules. Point-in-time-correct training is a deferred refinement.

In [22]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

# Training table: each transaction's charge attributes + its card's state + label.
session.sql(f"""
CREATE OR REPLACE TABLE {TRAIN_TABLE} AS
SELECT
  t.AMOUNT,
  CASE LOWER(t.MERCHANT_CATEGORY)
    WHEN 'cash withdrawal' THEN 0 WHEN 'electronics' THEN 1 WHEN 'entertainment' THEN 2
    WHEN 'fuel' THEN 3 WHEN 'groceries' THEN 4 WHEN 'jewelry' THEN 5
    WHEN 'restaurants' THEN 6 WHEN 'travel' THEN 7 ELSE -1 END AS MERCHANT_CATEGORY,
  t.LATITUDE, t.LONGITUDE,
  c.HOME_LAT, c.HOME_LON, c.LAST_LAT, c.LAST_LON, c.AVG_AMOUNT, c.TXN_COUNT,
  IFF(t.FRAUD_FLAG, 1, 0) AS IS_FRAUD
FROM {RAW_TABLE} t
JOIN {FEATURE_TABLE} c ON t.CARD_NUMBER = c.CARD_NUMBER
""").collect()

FEATURES = ["AMOUNT", "MERCHANT_CATEGORY", "LATITUDE", "LONGITUDE",
            "HOME_LAT", "HOME_LON", "LAST_LAT", "LAST_LON", "AVG_AMOUNT", "TXN_COUNT"]
TARGET = "IS_FRAUD"

data = session.table(TRAIN_TABLE).to_pandas()
print("Training rows:", len(data), "| fraud rate:", round(data[TARGET].mean(), 3))

X, y = data[FEATURES], data[TARGET]
stratify = y if y.nunique() > 1 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify)

pos = max((y_train == 1).sum(), 1)
model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                          scale_pos_weight=(y_train == 0).sum() / pos,
                          eval_metric="aucpr", random_state=42)
model.fit(X_train, y_train)

if y_test.nunique() > 1:
    print(f"PR-AUC: {average_precision_score(y_test, model.predict_proba(X_test)[:, 1]):.3f}")

Training rows: 222 | fraud rate: 0.054
PR-AUC: 0.833


In [23]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=FS_DATABASE, schema_name=FS_SCHEMA)

try:
    mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)
    print(f"Using existing model: {MODEL_NAME}/{MODEL_VERSION}")
except Exception:
    mv = reg.log_model(
        model_name=MODEL_NAME,
        version_name=MODEL_VERSION,
        model=model,
        sample_input_data=X_train.head(5),
        conda_dependencies=["xgboost"],
    )
    print(f"Model registered: {MODEL_NAME} version {MODEL_VERSION}")

print("Functions:", [f["name"] if isinstance(f, dict) else str(f) for f in mv.show_functions()])

/Users/ereitberger/dev_projects/credit_card_fraud_detection_demo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:03<00:07,  2.00s/it]  

/Users/ereitberger/dev_projects/credit_card_fraud_detection_demo/.venv/lib/python3.10/site-packages/snowflake/ml/registry/_manager/model_parameter_reconciler.py:84: UserWarning: `relax_version` is not set and therefore defaulted to True. Dependency version constraints relaxed from ==x.y.z to >=x.y, <(x+1). To use specific dependency versions for compatibility, reproducibility, etc., set `options={'relax_version': False}` when logging the model.
  reconciled_options = self._reconcile_relax_version(


Model logged successfully.: 100%|██████████| 6/6 [01:47<00:00, 17.96s/it]                          
Model registered: FRAUD_XGBOOST version V1
Functions: ['PREDICT', 'PREDICT_PROBA']


## Part V — Deploy real-time inference service (SPCS)

`feature_sources_per_function` maps `predict` to the Postgres-backed FeatureView, so external callers send only `TRANSACTION_ID` and the service auto-fetches features before scoring.

In [24]:
session.sql(f"""
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL_NAME}
  MIN_NODES = 1 MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_RESUME = TRUE AUTO_SUSPEND_SECS = 300
  COMMENT = 'Compute pool for fraud detection inference service'
""").collect()
session.sql(f"DESCRIBE COMPUTE POOL {COMPUTE_POOL_NAME}").show()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"name"              |"state"    |"min_nodes"  |"max_nodes"  |"instance_family"  |"num_services"  |"num_jobs"  |"auto_suspend_secs"  |"auto_resume"  |"active_nodes"  |"idle_nodes"  |"target_nodes"  |"created_on"                      |"resumed_on"                      |"updated_on"                      |"owner"       |"comment"                                           |"is_exclusive"  |"application"  |"error_code"  |"status_message"  |"placement_group"  |"backup_instance_families"  |
----------------------

In [25]:
card_fv = fs.get_feature_view(FEATURE_VIEW_NAME, FEATURE_VIEW_VERSION)

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL_NAME,
    ingress_enabled=True,
    feature_sources_per_function={"predict": [card_fv]},
)
print(f"Service deployment initiated: {SERVICE_NAME}")

create_service logs saved to: /Users/ereitberger/Library/Logs/snowflake-ml/model_deploy_8c554a66_1787664045.log    
To see logs in console, set log level to INFO: logging.getLogger().setLevel(logging.INFO)                     
Model service created successfully: 100%|██████████| 6/6 [05:31<00:00, 55.21s/it]                             
Service deployment initiated: FRAUD_DETECTION_SVC


In [26]:
print(mv.list_services().to_string())
# Once RUNNING, get the public ingress URL for your app to call:
session.sql(f"SHOW ENDPOINTS IN SERVICE {FS_DATABASE}.{FS_SCHEMA}.{SERVICE_NAME}").show()

                                         name     status                                         inference_endpoint                                       internal_endpoint  autocapture_enabled
0  FRAUD_ML.FEATURE_STORE.FRAUD_DETECTION_SVC  SUSPENDED  mn5jie-sfseeurope-ereitberger-aws2.snowflakecomputing.app  http://fraud-detection-svc.epat.svc.spcs.internal:5000                False
---------------------------------------------------------------------------------------------------------------------
|"name"     |"port"  |"port_range"  |"protocol"  |"is_public"  |"ingress_url"                                       |
---------------------------------------------------------------------------------------------------------------------
|inference  |5000    |NULL          |HTTP        |true         |mn5jie-sfseeurope-ereitberger-aws2.snowflakecom...  |
---------------------------------------------------------------------------------------------------------------------



## Part VI — How the app calls it

Before writing a transaction, the app POSTs the `CARD_NUMBER` plus the charge attributes to the ingress endpoint. The service looks up card state from the online store, fuses it with the charge, runs XGBoost, and returns the fraud probability.

```bash
curl -X POST "https://<INGRESS_URL>/predict" \
  -H 'Authorization: Snowflake Token="<PAT_TOKEN>"' \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {
        "columns": ["CARD_NUMBER", "AMOUNT", "MERCHANT_CATEGORY", "LATITUDE", "LONGITUDE"],
        "data": [["4539abc...", 1849.24, 5, 40.71, -74.01]]
      }}'
```

The request carries `CARD_NUMBER` (the lookup key) + the charge attributes; the service fills `HOME_LAT/LON`, `LAST_LAT/LON`, `AVG_AMOUNT`, `TXN_COUNT` from the online store. `MERCHANT_CATEGORY` is the integer code (see the mapping in Part IV).

**Auth setup for external clients** (run once): generate a PAT in Snowsight -> My Profile -> Authentication -> Personal Access Tokens, and grant ingress OAuth:
```sql
ALTER USER <your_user> ADD DELEGATED AUTHORIZATION
  OF ROLE ACCOUNTADMIN TO SECURITY INTEGRATION SNOWSERVICES_INGRESS_OAUTH;
```
If the account has a network policy, whitelist the caller's IP. The returned probability drives `fraud_flag` before the transaction is written to Postgres.

> Note: `mv.run()` (Python SDK) and SQL service functions bypass the ingress gateway and do **not** trigger feature enrichment. Auto-lookup is only for the external REST endpoint.

## Cleanup (optional)

Tears down the ML infra. **Leave the mirror alone** unless you truly want to stop replication — dropping it removes the target DB and the 7-day change feed.

In [ ]:
# ML infra teardown
session.sql(f"DROP SERVICE IF EXISTS {FS_DATABASE}.{FS_SCHEMA}.{SERVICE_NAME}").collect()
session.sql(f"DROP MODEL IF EXISTS {FS_DATABASE}.{FS_SCHEMA}.{MODEL_NAME}").collect()
session.sql(f"DROP COMPUTE POOL IF EXISTS {COMPUTE_POOL_NAME}").collect()
print("ML infra dropped.")

# The mirror (FRAUD_DEMO_OLTP) is managed outside this notebook -- do not drop it here.